# 1. Xóa duplicates 

In [1]:
from pathlib import Path
DATASET_PATH_2 = Path("/Users/mac/MeterReadAI/data/DataBeforeHandle/Word-Wheel_Water_Meter_Dataset/recognition/recognition_5-digit/train/5-digit_train_img")

dataset_imgs = list(DATASET_PATH_2.glob("*.*"))
print("Bộ dữ liệu train có: ", len(dataset_imgs))

Bộ dữ liệu train có:  11867


In [6]:
import hashlib
from collections import defaultdict

def check_duplicates(data_path, delete=False):
    hash_dict = defaultdict(list)
    for img_path in data_path.glob("*.*"):
        with open(img_path, "rb") as f:
            file_hash = hashlib.md5(f.read()).hexdigest()
            hash_dict[file_hash].append(img_path)

    duplicates = {h: files for h, files in hash_dict.items() if len(files) > 1}

    if duplicates:
        remove = 0
        print("Phát hiện các file trùng lặp chính xác:")
        for h, files in duplicates.items():
            print(f"- Nhóm trùng ({h}): {files}")
            if delete:
                for img_path in files[1:]:
                    img_path.unlink()
                    remove += 1
        if delete:
            print(f"\nHoàn tất! Đã xóa tổng cộng {remove} file trùng lặp.")
        else:
            print(f"\nĐang ở chế độ xem. Truyền tham số delete=True để tiến hành xóa")
    else:
        print("Không tìm thấy file nào trùng lặp chính xác.")
check_duplicates(DATASET_PATH_2, delete=True)

Phát hiện các file trùng lặp chính xác:
- Nhóm trùng (96f7a42de0d8e0685282043c8aeb89d5): [PosixPath('/Users/mac/MeterReadAI/data/DataBeforeHandle/Word-Wheel_Water_Meter_Dataset/recognition/recognition_5-digit/train/5-digit_train_img/train8744.jpg'), PosixPath('/Users/mac/MeterReadAI/data/DataBeforeHandle/Word-Wheel_Water_Meter_Dataset/recognition/recognition_5-digit/train/5-digit_train_img/train8747.jpg')]
- Nhóm trùng (99533d1697274240cedbac1ec422db8e): [PosixPath('/Users/mac/MeterReadAI/data/DataBeforeHandle/Word-Wheel_Water_Meter_Dataset/recognition/recognition_5-digit/train/5-digit_train_img/train3956.jpg'), PosixPath('/Users/mac/MeterReadAI/data/DataBeforeHandle/Word-Wheel_Water_Meter_Dataset/recognition/recognition_5-digit/train/5-digit_train_img/train3957.jpg')]
- Nhóm trùng (302aad8861424d9a0031a03b3a2ae46d): [PosixPath('/Users/mac/MeterReadAI/data/DataBeforeHandle/Word-Wheel_Water_Meter_Dataset/recognition/recognition_5-digit/train/5-digit_train_img/train8369.jpg'), PosixPath(

In [7]:
check_duplicates(DATASET_PATH_2, delete=False)

Không tìm thấy file nào trùng lặp chính xác.


# 2. Kiểm tra duplicate trên train và test

In [8]:
import hashlib
from pathlib import Path

def check_train_test_leakage(train_dir, test_dir):
    train_hashes = {}
    
    for img_path in Path(train_dir).glob("*.*"):
        with open(img_path, "rb") as f:
            file_hash = hashlib.md5(f.read()).hexdigest()
            train_hashes[file_hash] = img_path.name

    leakage_count = 0
    
    for test_path in Path(test_dir).glob("*.*"):
        with open(test_path, "rb") as f:
            test_hash = hashlib.md5(f.read()).hexdigest()
            
        if test_hash in train_hashes:
            leakage_count += 1
            train_file = train_hashes[test_hash]
            print(f"[Phát hiện rò rỉ] Ảnh test '{test_path.name}' trùng lặp với ảnh train '{train_file}'")

    if leakage_count > 0:
        print(f"\nCảnh báo: Tìm thấy tổng cộng {leakage_count} file ở tập test bị trùng với tập train!")
    else:
        print("\nTuyệt vời! Không có sự trùng lặp nào giữa tập train và tập test.")

check_train_test_leakage("/Users/mac/MeterReadAI/data/DataBeforeHandle/Word-Wheel_Water_Meter_Dataset/recognition/recognition_5-digit/train/5-digit_train_img", "/Users/mac/MeterReadAI/data/DataBeforeHandle/Word-Wheel_Water_Meter_Dataset/recognition/recognition_5-digit/test/5-digit_test_img")


Tuyệt vời! Không có sự trùng lặp nào giữa tập train và tập test.


In [9]:
check_train_test_leakage("/Users/mac/MeterReadAI/data/DataBeforeHandle/Word-Wheel_Water_Meter_Dataset/recognition/recognition_6-digit/train/6-digit_train_img", "/Users/mac/MeterReadAI/data/DataBeforeHandle/Word-Wheel_Water_Meter_Dataset/recognition/recognition_6-digit/test/6-digit_test_img")


Tuyệt vời! Không có sự trùng lặp nào giữa tập train và tập test.


# 3. Kiểm tra dataset sau khi merge xem file csv có khớp với từng ảnh 

In [1]:
import os
import pandas as pd
from pathlib import Path

def check_dataset_sync(base_dir, split="train"):
    base_path = Path(base_dir) / split
    img_dir = base_path / "images"
    csv_path = base_path / f"{split}_labels.csv"
    
    print(f"=== Đang kiểm tra phân vùng: {split.upper()} ===")
    
    if not csv_path.exists():
        print(f"Không tìm thấy file CSV tại: {csv_path}")
        return
    if not img_dir.exists():
        print(f"Không tìm thấy thư mục ảnh tại: {img_dir}")
        return

    # Đọc file CSV
    df = pd.read_csv(csv_path)
    print(f"Các cột trong CSV: {list(df.columns)}")
    
    img_col = df.columns[0] 
    print(f"Đang đối chiếu dựa trên cột: '{img_col}'")

    csv_images = set(df[img_col].astype(str))
    
    disk_images = {f for f in os.listdir(img_dir) if not f.startswith('.')}

    # Kiểm tra chéo
    missing_on_disk = csv_images - disk_images
    missing_in_csv = disk_images - csv_images

    print(f"- Số dòng trong CSV: {len(df)}")
    print(f"- Số ảnh thực tế trong thư mục: {len(disk_images)}")
    
    if missing_on_disk:
        print(f"⚠️ Có {len(missing_on_disk)} ảnh có tên trong CSV nhưng KHÔNG TỒN TẠI trong thư mục:")
        print(list(missing_on_disk)[:5]) # Hiển thị tối đa 5 lỗi mẫu
    else:
        print("✅ Tuyệt vời! Tất cả ảnh trong CSV đều tồn tại trên ổ đĩa.")

    if missing_in_csv:
        print(f"⚠️ Có {len(missing_in_csv)} ảnh trong thư mục nhưng KHÔNG CÓ TRONG CSV:")
        print(list(missing_in_csv)[:5]) # Hiển thị tối đa 5 lỗi mẫu
    else:
        print("✅ Tuyệt vời! Không có ảnh thừa nào trên ổ đĩa thiếu trong CSV.")
    print("\n")

# Đường dẫn đến thư mục recognition_merge của bạn
base_directory = "/Users/mac/MeterReadAI/data/DataForTrain/recognition_merge"

# Chạy kiểm tra cho cả train và test
check_dataset_sync(base_directory, "train")
check_dataset_sync(base_directory, "test")

=== Đang kiểm tra phân vùng: TRAIN ===
Các cột trong CSV: ['filename', 'label']
Đang đối chiếu dựa trên cột: 'filename'
- Số dòng trong CSV: 14387
- Số ảnh thực tế trong thư mục: 14387
✅ Tuyệt vời! Tất cả ảnh trong CSV đều tồn tại trên ổ đĩa.
✅ Tuyệt vời! Không có ảnh thừa nào trên ổ đĩa thiếu trong CSV.


=== Đang kiểm tra phân vùng: TEST ===
Các cột trong CSV: ['filename', 'label']
Đang đối chiếu dựa trên cột: 'filename'
- Số dòng trong CSV: 2400
- Số ảnh thực tế trong thư mục: 2400
✅ Tuyệt vời! Tất cả ảnh trong CSV đều tồn tại trên ổ đĩa.
✅ Tuyệt vời! Không có ảnh thừa nào trên ổ đĩa thiếu trong CSV.


